# GP04 — Explore GeoPackage

Ανοίγω → Εξερευνώ → Κατανοώ → Φιλτράρω → Επαληθεύω → Εξάγω

In [ ]:
import geopandas as gpd
from pathlib import Path
gpkg_path = Path("data/gp04/serres_assets.gpkg")
assert gpkg_path.exists()
layers = gpd.list_layers(gpkg_path)
print(layers)

## 1. Layers και επιλογή

In [ ]:
assert {"parks", "water_points"}.issubset(set(layers["name"]))
parks = gpd.read_file(gpkg_path, layer="parks")
water = gpd.read_file(gpkg_path, layer="water_points")
print("Parks rows:", len(parks))
print("Water rows:", len(water))

## 2. Schema, CRS και geometry

In [ ]:
print(parks.dtypes)
print(water.dtypes)
print("Parks CRS/types:", parks.crs, parks.geometry.geom_type.value_counts().to_dict())
print("Water CRS/types:", water.crs, water.geometry.geom_type.value_counts().to_dict())
print("Missing parks/water:", parks.isna().sum().sum(), water.isna().sum().sum())

## 3. Bounds και χάρτης

In [ ]:
print("Parks bounds:", parks.total_bounds)
print("Water bounds:", water.total_bounds)
ax = parks.plot(figsize=(7, 6), color="green", edgecolor="black")
water.plot(ax=ax, color="blue", markersize=50)

## 4. Layer-specific GeoPython QA

In [ ]:
assert len(parks) == 3 and len(water) == 3
assert parks.crs is not None and water.crs is not None
assert parks.crs == water.crs
assert parks.isna().sum().sum() == 0 and water.isna().sum().sum() == 0
assert parks.geometry.geom_type.eq("Polygon").all()
assert water.geometry.geom_type.eq("Point").all()
assert parks.geometry.notna().all() and water.geometry.notna().all()
assert parks.geometry.is_empty.sum() == 0 and water.geometry.is_empty.sum() == 0
assert parks.geometry.is_valid.all() and water.geometry.is_valid.all()
assert parks.total_bounds.shape == (4,)
print("Όλα τα layer QA checks πέρασαν.")

## Mini Challenge — Ενδεικτική λύση

In [ ]:
active_water = water[water["status"] == "active"].copy()
assert len(active_water) == 2
assert active_water.geometry.geom_type.eq("Point").all()
assert active_water.crs.to_epsg() == 4326
output_dir = Path("outputs/gp04")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "active_water_points.geojson"
active_water.to_file(output_path, driver="GeoJSON")
assert len(gpd.read_file(output_path)) == 2
print("Challenge export OK")

## Τελικός έλεγχος

Το GeoPackage εξερευνήθηκε layer προς layer και το αποτέλεσμα επαληθεύτηκε με αριθμητικούς και οπτικούς ελέγχους.